In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from DATA.stock_invest_function import *

In [3]:
TABLE_NAME = "korea_monthly_trade_forecast_v2"

def make_pivot_from_trade_forecast_db(
    engine,
    forecast_dates: list,
    indicator: str,
    table_name: str = TABLE_NAME,
    aggfunc: str = "last"   # 중복 시 처리: "last" / "sum" 등
) -> pd.DataFrame:
    """
    forecast_dates(리스트) + indicator 조건으로 DB에서 데이터 조회 후
    index=date, columns=hs_code, values=value pivot 생성
    """

    if not forecast_dates:
        raise ValueError("forecast_dates 리스트가 비어 있습니다.")

    # MySQL IN 절용 placeholder 생성
    placeholders = ", ".join([f":d{i}" for i in range(len(forecast_dates))])
    params = {f"d{i}": forecast_dates[i] for i in range(len(forecast_dates))}
    params["indicator"] = indicator

    sql = text(f"""
        SELECT date, hs_code, value
        FROM {table_name}
        WHERE forecast_date IN ({placeholders})
          AND indicator = :indicator
    """)

    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params=params)

    if df.empty:
        raise ValueError("조건에 해당하는 데이터가 없습니다. forecast_dates/indicator를 확인하세요.")

    # 타입 정리
    df["date"] = pd.to_datetime(df["date"])
    df["hs_code"] = df["hs_code"].astype(str)

    # pivot
    pivot_df = (
        df.pivot_table(
            index="date",
            columns="hs_code",
            values="value",
            aggfunc=aggfunc  # 중복 시 마지막 값/합계 등 선택
        )
        .sort_index()
    )

    return pivot_df

In [2]:
db_info = {
    'host': get_db_host(),
    # 'host' : 'hystox74.synology.me',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

query = text("""
    SELECT DISTINCT forecast_date
    FROM korea_monthly_trade_forecast_v2
    ORDER BY forecast_date
""")

with engine.connect() as conn:
    forecast_dates = pd.read_sql(query, conn)

print(forecast_dates)

  forecast_date
0    2025-10-31
1    2025-11-02
2    2025-11-03
3    2025-11-11
4    2025-11-12
5    2025-11-13
6    2025-12-09
7    2025-12-10
8    2025-12-12
9    2025-12-13


In [48]:
forecast_dates = ["2025-11-12", "2025-11-13", "2025-11-14"]
indicator = "sarima_expDlr"

last_month_pivot_df = make_pivot_from_trade_forecast_db(
    engine=engine,
    forecast_dates=forecast_dates,
    indicator=indicator,
    aggfunc="last"
)



In [44]:
target_date = "2027-08-31"

# index가 datetime인 경우를 가정
target_date = pd.to_datetime(target_date)

# 해당 날짜에서 NaN인 컬럼 drop
df_filtered = last_month_pivot_df.loc[:, ~last_month_pivot_df.loc[target_date].isna()]

last_month_forecast = df_filtered.loc['2025-11':]

In [53]:
last_month_forecast

hs_code,1201,1515500000,1515901000,1604,1703,190110,1902,190230,1902301010,19049,...,9030820000,903090,9030901000,903149,9031499000,9031809091,940340,9405,961610,961900
date,,,,,,,,,,,,,,,,,,,,,
2025-11-30,4900.036585,1.478510e+06,270525.079782,1.043194e+07,4957.377799,7.572944e+06,1.580400e+08,1.468811e+08,1.380395e+08,9.864805e+06,...,2.182716e+07,6.094851e+07,5.666145e+07,1.547634e+08,1.487225e+08,1.991142e+06,374923.295875,1.305769e+07,216836.462120,4.000204e+06
2025-12-31,12724.783352,1.293808e+06,261485.583477,9.782043e+06,6027.453315,7.021328e+06,1.539846e+08,1.436037e+08,1.360929e+08,8.472869e+06,...,2.415866e+07,5.550577e+07,5.243951e+07,1.865985e+08,1.734799e+08,2.315234e+06,383107.470850,1.495298e+07,333266.763541,5.084521e+06
2026-01-31,-1452.730047,1.362301e+06,241349.041666,8.432631e+06,5958.328134,5.065200e+06,1.456442e+08,1.368190e+08,1.291437e+08,7.360805e+06,...,2.292261e+07,5.526741e+07,5.188655e+07,1.742006e+08,1.623959e+08,1.624208e+06,295140.529995,1.024466e+07,-21231.318949,4.696518e+06
2026-02-28,17420.577723,1.266147e+06,230699.916132,9.835437e+06,4874.458508,6.398115e+06,1.607632e+08,1.493484e+08,1.406527e+08,8.834736e+06,...,2.541112e+07,5.640613e+07,5.407591e+07,1.716006e+08,1.561107e+08,1.805570e+06,150507.165244,9.738012e+06,-3344.696780,4.805501e+06
2026-03-31,4860.742279,1.361816e+06,217065.626545,1.021116e+07,4479.143475,6.985428e+06,1.610736e+08,1.493863e+08,1.389979e+08,8.727004e+06,...,2.959247e+07,5.689936e+07,5.473788e+07,1.902575e+08,1.729561e+08,2.043453e+06,224641.440318,1.355518e+07,-2780.660979,4.794969e+06
2026-04-30,14126.469481,1.439958e+06,238010.013805,1.034003e+07,3917.287958,5.548693e+06,1.739661e+08,1.614541e+08,1.492723e+08,9.046207e+06,...,3.048521e+07,5.777708e+07,5.529083e+07,1.885406e+08,1.700587e+08,1.802997e+06,161862.450948,1.401682e+07,-17639.061628,6.883072e+06
2026-05-31,4039.054216,1.375804e+06,243364.674251,9.236419e+06,6006.751579,5.180452e+06,1.674289e+08,1.563320e+08,1.458708e+08,9.394435e+06,...,2.804766e+07,5.964897e+07,5.710903e+07,1.895938e+08,1.716705e+08,2.615067e+06,255827.786507,1.335758e+07,66413.656568,4.732340e+06
2026-06-30,11873.022087,1.516427e+06,246424.882898,9.284481e+06,4458.207812,5.654460e+06,1.685857e+08,1.578589e+08,1.474741e+08,9.531171e+06,...,2.823103e+07,6.267621e+07,6.062635e+07,1.827897e+08,1.621375e+08,1.601381e+06,193316.500347,1.284632e+07,146678.730102,4.820561e+06
2026-07-31,3416.172995,1.471220e+06,221218.029032,9.296220e+06,3869.916742,4.852149e+06,1.666897e+08,1.555282e+08,1.464103e+08,9.423976e+06,...,2.660091e+07,6.397029e+07,6.128066e+07,1.761282e+08,1.556120e+08,1.659432e+06,252713.379972,1.497987e+07,111147.081300,5.312821e+06


In [51]:
forecast_dates = ["2025-12-12", "2025-12-13"]
indicator = "sarima_expDlr"

this_month_pivot_df = make_pivot_from_trade_forecast_db(
    engine=engine,
    forecast_dates=forecast_dates,
    indicator=indicator,
    aggfunc="last"
)

target_date = "2027-08-31"

# index가 datetime인 경우를 가정
target_date = pd.to_datetime(target_date)

# 해당 날짜에서 NaN인 컬럼 drop
this_month_df_filtered = this_month_pivot_df.loc[:, ~this_month_pivot_df.loc[target_date].isna()]

this_month_forecast = this_month_df_filtered.loc['2025-12':]



In [52]:
this_month_forecast['854232']

date
2025-12-31    1.105064e+10
2026-01-31    9.772243e+09
2026-02-28    1.012278e+10
2026-03-31    1.144964e+10
2026-04-30    1.059402e+10
2026-05-31    1.163905e+10
2026-06-30    1.270374e+10
2026-07-31    1.180984e+10
2026-08-31    1.236403e+10
2026-09-30    1.314320e+10
2026-10-31    1.253517e+10
2026-11-30    1.310977e+10
2026-12-31    1.397158e+10
2027-01-31    1.235906e+10
2027-02-28    1.258671e+10
2027-03-31    1.400543e+10
2027-04-30    1.310252e+10
2027-05-31    1.422778e+10
2027-06-30    1.523515e+10
2027-07-31    1.436875e+10
2027-08-31    1.498367e+10
2027-09-30    1.576020e+10
2027-10-31    1.517174e+10
2027-11-30    1.587788e+10
Name: 854232, dtype: float64